# text8 GPT — Analysis

A decoder-only Transformer (grouped-query attention + RoPE + QK-norm) trained on
text8, tokenized with the pretrained [LiquidAI/LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M) subword tokenizer.

Training now happens in `train.py`:

    uv run python projects/text8/gpt/train.py

This notebook only loads the resulting checkpoint and metrics for analysis:
a data preview, training curves, and text generation.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import glob
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from chimera.data import Text8DataModule
from chimera.models import GPT

RUN_DIR = "/mnt/ai/runs/text8/gpt"
DATA_DIR = "/mnt/ai/data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Must match the GPT config trained in train.py.
SEQ_LEN = 2048
N_EMBD = 48
N_HEAD = 2
N_KV_HEAD = 1
N_LAYER = 6

## Data

Load text8 and tokenize it with the pretrained **LiquidAI/LFM2.5-230M** subword tokenizer (vocab ≈ 64k) instead of the classic 27-symbol character vocabulary.

In [ ]:
dm = Text8DataModule(
    data_dir=DATA_DIR,
    batch_size=32,
    seq_len=SEQ_LEN,
    num_workers=0,
    pin_memory=False,
    tokenizer_backend="pretrained",  # LiquidAI/LFM2.5-230M
)
dm.prepare_data()
dm.setup("fit")

train_loader = dm.train_dataloader()

print(f"tokenizer={dm.pretrained_id}  vocab_size={dm.vocab_size}")
print(f"train sequences={len(dm.text8_train):,}  val sequences={len(dm.text8_val):,}")

In [ ]:
x, y = next(iter(train_loader))
print("sample:", repr(dm.decode(x[0][:64])))

# most frequent tokens in one batch, shown as their decoded text
counts = Counter(x.flatten().tolist())
top = counts.most_common(20)
labels = [repr(dm.decode([tid])) for tid, _ in top]
values = [c for _, c in top]

plt.figure(figsize=(12, 4))
plt.bar(range(len(top)), values)
plt.title("Top-20 Tokens (one training batch)")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(range(len(top)), labels, rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Model

Rebuild the GPT with the same config as `train.py` and load its checkpoint.
`block_size` must match the datamodule's `seq_len`.

In [ ]:
# round to multiple of 32 (must match train.py)
rounded_vocab_size = (dm.vocab_size + 31) // 32 * 32

model = GPT(
    vocab_size=rounded_vocab_size,
    block_size=SEQ_LEN,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    n_layer=N_LAYER,
    tie_embedding=True,
)

ckpt = torch.load(f"{RUN_DIR}/checkpoints/gpt.ckpt", map_location="cpu", weights_only=False)
# train.py compiles the model before wrapping it, so checkpoint keys look like
# "model._orig_mod.<...>"; strip both prefixes to load into the eager GPT.
state = {}
for k, v in ckpt["state_dict"].items():
    if not k.startswith("model."):
        continue
    state[k.removeprefix("model.").removeprefix("_orig_mod.")] = v
model.load_state_dict(state)
model.to(DEVICE).eval()
print(f"loaded {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params "
      f"from {RUN_DIR}/checkpoints/gpt.ckpt")

## Analysis

Plot the logged loss and bits-per-token curves, then sample text from the trained model.

In [ ]:
import glob

# CSVLogger writes to RUN_DIR/csv/version_*/metrics.csv; take the latest run.
csv_path = sorted(glob.glob(f"{RUN_DIR}/csv/version_*/metrics.csv"))[-1]
metrics = np.genfromtxt(csv_path, delimiter=",", names=True)


def series(step_key, value_key):
    step = metrics[step_key]
    value = metrics[value_key]
    mask = ~np.isnan(value)
    return step[mask], value[mask]


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Training Metrics")

for ax, metric, title in zip(axes, ["loss", "bpt"], ["Loss (nats)", "Bits per Token"]):
    train_step, train_val = series("step", f"train{metric}")
    val_step, val_val = series("step", f"val{metric}")
    ax.plot(train_step, train_val, marker="o", label="train")
    ax.plot(val_step, val_val, marker="o", label="val")
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.show()

In [ ]:
from textwrap import fill

prompt = "the meaning of life is "
idx = torch.tensor([dm.tokenizer.encode(prompt)], device=DEVICE)

# Pass compile=True for ~2x faster decode when generating repeatedly (one-time
# ~10s compile on the first call); eager by default here.
generated = model.generate(idx, max_new_tokens=450, temperature=0.8)
print("\n".join(fill(dm.decode(generated[0].cpu()), width=80).splitlines()))